# NB_patient_360

Builds gold table gold.patient_360` (one row per patient) from silver patients, hospitals, insurance_providers and aggregated lab_results, per the Silver-to-Gold STTM. Load type: full rebuild (CREATE OR REPLACE TABLE ... AS SELECT); all joins are LEFT JOINs from `silver.patients`.

In [0]:
dbutils.widgets.text("catalog", "adb_ws_wecare", "Catalog")
catalog = dbutils.widgets.get("catalog")
silver, gold = f"{catalog}.silver", f"{catalog}.gold"
target = f"{gold}.patient_360"
print(f"Target: {target}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")

spark.sql(f"""
CREATE OR REPLACE TABLE {target}
COMMENT 'Gold: one row per patient with hospital, insurance and lab-test summary (STTM Silver to Gold)'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                  AS total_tests,
    COUNT_IF(result_status = 'Abnormal')      AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')      AS critical_test_count,
    MAX(test_date)                            AS last_test_date
  FROM {silver}.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                    AS patient_id,
  CAST(p.first_name AS STRING)                                                 AS first_name,
  CAST(p.last_name AS STRING)                                                  AS last_name,
  CAST(p.date_of_birth AS DATE)                                                AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  CAST(p.gender AS STRING)                                                     AS gender,
  CAST(p.city AS STRING)                                                       AS city,
  CAST(p.state AS STRING)                                                      AS state,
  CAST(p.registered_hospital_id AS INT)                                        AS hospital_id,
  CAST(h.hospital_name AS STRING)                                              AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                         AS insurance_provider_id,
  CAST(ip.provider_name AS STRING)                                             AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                      AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                              AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                              AS critical_test_count,
  CAST(l.last_test_date AS DATE)                                               AS last_test_date
FROM {silver}.patients p
LEFT JOIN {silver}.hospitals h            ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {silver}.insurance_providers ip ON p.insurance_provider_id  = ip.insurance_provider_id
LEFT JOIN lab_agg l                       ON p.patient_id             = l.patient_id
""")
print(f"Rebuilt {target}")

In [0]:
updated_df = spark.table(target)
record_count = updated_df.count()
print(f"{target}: {record_count} records")
dbutils.notebook.exit(str(record_count))